# 04 — Frontier-Vergleich: ChatGPT vs. Claude

Dieses Notebook vergleicht eure Hand-Annotation aus Phase 2 (`annotation/meine_gold.csv`) mit zwei Frontier-LLM-Annotationen derselben 12 Anzeigen:

- `frontier_predictions_chatgpt.json`
- `frontier_predictions_claude.json`

Output: zwei Frontier-CSV-Dateien, eine κ-Tabelle pro Modell sowie direkte Modell-Differenzen.


## Run-Header

| Feld | Wert |
|---|---|
| Datum | _YYYY-MM-DD_ |
| Frontier-Modell 1 | ChatGPT |
| Frontier-Modell 2 | Claude |
| Prompt-Variante | _z. B. gleicher Prompt mit 3 Few-Shots_ |
| Anzahl Korrektur-Turns | _ |
| Schema-Verletzungs-Mapping | _Werte, die gemappt wurden_ |
| Auffälligkeiten | _ |
| ChatGPT-JSON | `frontier_predictions_chatgpt.json` |
| Claude-JSON | `frontier_predictions_claude.json` |
| ChatGPT-CSV | `annotation/frontier_gold_chatgpt.csv` |
| Claude-CSV | `annotation/frontier_gold_claude.csv` |
| Eigene Gold-CSV | `annotation/meine_gold.csv` |


## Daten laden + Schema-Konformität prüfen

Die Zellen sind so gebaut, dass die beiden Prediction-JSONs im Notebook-Ordner, im Repo-Root oder im Ordner `annotation/` liegen dürfen.


In [1]:
import csv
import json
import re
from collections import Counter
from pathlib import Path
from textwrap import dedent

import pandas as pd

# ── Robuste Pfade ─────────────────────────────────────────────────────────
# Das Notebook kann aus /notebooks, dem Repo-Root oder einer Notebook-UI laufen.
CWD = Path.cwd().resolve()
SEARCH_DIRS = [
    CWD,
    CWD.parent,
    CWD / "annotation",
    CWD.parent / "annotation",
    CWD / "daten",
    CWD.parent / "daten",
    Path("/mnt/data"),  # nur relevant in ChatGPT/Sandbox, lokal harmlos
]

def find_existing_file(*names, required=False):
    """Suche eine Datei in typischen Repo-/Notebook-Ordnern."""
    candidates = []
    for name in names:
        p = Path(name)
        candidates.append(p)
        if p.is_absolute() and p.exists():
            return p
        for base in SEARCH_DIRS:
            candidates.append(base / name)
            if (base / name).exists():
                return (base / name).resolve()
    if required:
        msg = "Datei nicht gefunden. Geprüfte Kandidaten:\n" + "\n".join(f"  - {p}" for p in candidates)
        raise FileNotFoundError(msg)
    return Path(names[0])

GOLD_PATH = find_existing_file(
    "meine_gold.csv",
    "annotation/meine_gold.csv",
    "../annotation/meine_gold.csv",
)
KORPUS_PATH = find_existing_file(
    "eigener_korpus.jsonl",
    "daten/eigener_korpus.jsonl",
    "../daten/eigener_korpus.jsonl",
)

PREDICTION_FILES = {
    "chatgpt": find_existing_file(
        "frontier_predictions_chatgpt.json",
        "annotation/frontier_predictions_chatgpt.json",
        "../frontier_predictions_chatgpt.json",
    ),
    "claude": find_existing_file(
        "frontier_predictions_claude.json",
        "annotation/frontier_predictions_claude.json",
        "../frontier_predictions_claude.json",
    ),
}

# CSV-Ausgaben: bevorzugt in denselben annotation-Ordner wie meine_gold.csv,
# sonst in ./annotation anlegen.
if GOLD_PATH.exists():
    ANNOTATION_DIR = GOLD_PATH.parent
else:
    ANNOTATION_DIR = (CWD.parent / "annotation") if (CWD.parent / "annotation").exists() else (CWD / "annotation")
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

FRONTIER_CSV_PATHS = {
    "chatgpt": ANNOTATION_DIR / "frontier_gold_chatgpt.csv",
    "claude": ANNOTATION_DIR / "frontier_gold_claude.csv",
}

print("Gefundene Dateien:")
print(f"  Gold:   {GOLD_PATH if GOLD_PATH.exists() else 'NICHT GEFUNDEN'}")
print(f"  Korpus: {KORPUS_PATH if KORPUS_PATH.exists() else 'NICHT GEFUNDEN'}")
for model, path in PREDICTION_FILES.items():
    print(f"  {model}: {path if path.exists() else 'NICHT GEFUNDEN'}")

# Gold/Korpus nur laden, wenn vorhanden. Für reine JSON→CSV-Konvertierung sind sie nicht nötig.
gold_df = pd.read_csv(GOLD_PATH) if GOLD_PATH.exists() else None
korpus = pd.read_json(KORPUS_PATH, lines=True) if KORPUS_PATH.exists() else None

if gold_df is not None:
    gold_ids = gold_df["id"].astype(str).tolist()
    print(f"\nGold-Records geladen: {len(gold_df)}")
else:
    gold_ids = []
    print("\nHinweis: Ohne annotation/meine_gold.csv wird nur JSON→CSV erzeugt; κ-Vergleich wird übersprungen.")

if korpus is not None and gold_ids:
    anzeigen = korpus.set_index("refnr").loc[gold_ids]
else:
    anzeigen = None


<jemalloc>: Unsupported system page size


Gefundene Dateien:
  Gold:   /home/jovyan/work/notebooks/Maschine learning/LLM-Workshop/annotation/meine_gold.csv
  Korpus: /home/jovyan/work/notebooks/Maschine learning/LLM-Workshop/daten/eigener_korpus.jsonl
  chatgpt: /home/jovyan/work/notebooks/Maschine learning/LLM-Workshop/annotation/frontier_predictions_chatgpt.json
  claude: /home/jovyan/work/notebooks/Maschine learning/LLM-Workshop/annotation/frontier_predictions_claude.json

Gold-Records geladen: 12


In [2]:
# Beide Prediction-JSONs → annotation/frontier_gold_chatgpt.csv und annotation/frontier_gold_claude.csv

SCHEMA_VERLETZUNG_MAP = {
    "homeoffice": {
        "möglich": "teilweise", "moeglich": "teilweise",
        "nach absprache": "teilweise", "flexibel": "teilweise",
        "mobiles arbeiten": "teilweise", "hybrid": "teilweise",
        "kein homeoffice": "nein", "vollzeit remote": "remote",
        "100% remote": "remote", "100 % remote": "remote",
        "vollständig remote": "remote", "ortsunabhängig": "remote",
    },
    "vertragsart": {
        "freelance": "sonstiges", "freiberuflich": "sonstiges",
        "selbstaendig": "sonstiges", "selbständig": "sonstiges",
        "lehrbeauftragter": "sonstiges", "lehrbeauftragte": "sonstiges",
        "leiharbeit": "sonstiges", "trainee": "sonstiges",
    },
    "erfahrungslevel": {
        "berufseinsteiger": "junior", "entry": "junior", "anfänger": "junior",
        "mittel": "mid", "intermediate": "mid",
        "expert": "senior", "experte": "senior",
        "alle level": "egal", "egal welches level": "egal",
    },
    "gehalt_zeitraum": {
        "month": "monat", "monatlich": "monat", "monatsgehalt": "monat",
        "year": "jahr", "jährlich": "jahr", "jaehrlich": "jahr", "jahresgehalt": "jahr",
    },
}

VALID_VALUES = {
    "homeoffice": {"ja", "teilweise", "nein", "remote", "nicht_genannt"},
    "vertragsart": {"ausbildung", "festanstellung", "praktikum", "werkstudent", "sonstiges"},
    "erfahrungslevel": {"junior", "mid", "senior", "egal", "nicht_genannt"},
    "gehalt_zeitraum": {"monat", "jahr"},
}

FELDER_CSV = [
    "id", "homeoffice", "vertragsart", "erfahrungslevel",
    "gehalt_min_eur", "gehalt_zeitraum", "skills_top3", "notiz"
]

def extract_json_array(path: Path):
    raw = path.read_text(encoding="utf-8").strip()
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        # Falls ein Modell doch Begleittext geschrieben hat: erstes JSON-Array herauspulen
        match = re.search(r"\[.*\]", raw, re.DOTALL)
        if not match:
            raise ValueError(f"Kein JSON-Array in {path} gefunden — Output prüfen.")
        data = json.loads(match.group(0))
    if not isinstance(data, list):
        raise TypeError(f"{path} enthält kein JSON-Array.")
    return data

def map_value(feld, val):
    if val is None:
        return None
    if isinstance(val, (list, int, float)):
        return val
    s = str(val).strip()
    s_lower = s.lower()
    return SCHEMA_VERLETZUNG_MAP.get(feld, {}).get(s_lower, s_lower)

def normalize_salary(v):
    if v is None or v == "":
        return ""
    try:
        return int(float(str(v).replace(".", "").replace(",", ".")))
    except ValueError:
        return str(v).strip()

def normalize_skills(v):
    if v is None:
        return ""
    if isinstance(v, list):
        return "|".join(str(x).strip() for x in v[:3] if str(x).strip())
    return str(v).strip()

def records_to_csv_rows(records):
    mapped_count = 0
    csv_rows = []
    problems = []

    for idx, rec in enumerate(records, start=1):
        if not isinstance(rec, dict):
            problems.append(f"Record {idx}: kein Objekt")
            continue

        rid = rec.get("id")
        row = {"id": rid, "notiz": ""}

        for feld in ["homeoffice", "vertragsart", "erfahrungslevel", "gehalt_zeitraum"]:
            original = rec.get(feld)
            mapped = map_value(feld, original)

            if isinstance(original, str) and original.strip().lower() != str(mapped).lower():
                mapped_count += 1

            # Wenn Gehalt fehlt, muss Zeitraum leer sein.
            if feld == "gehalt_zeitraum" and rec.get("gehalt_min_eur") in (None, ""):
                mapped = None

            if mapped is not None and feld in VALID_VALUES and mapped not in VALID_VALUES[feld]:
                problems.append(f"{rid}: ungültiger Wert {feld}={mapped!r}")

            row[feld] = "" if mapped is None else mapped

        row["gehalt_min_eur"] = normalize_salary(rec.get("gehalt_min_eur"))
        row["skills_top3"] = normalize_skills(rec.get("skills_top3"))
        csv_rows.append(row)

    return csv_rows, mapped_count, problems

all_frontier_dfs = {}

for model, json_path in PREDICTION_FILES.items():
    if not json_path.exists():
        raise FileNotFoundError(
            f"Prediction-Datei für {model!r} fehlt: {json_path}\n"
            f"Lege sie als frontier_predictions_{model}.json in den Notebook-Ordner, Repo-Root oder annotation/."
        )

    records = extract_json_array(json_path)
    print(f"\n{model}: Records geladen: {len(records)} (erwartet: 12)")

    csv_rows, mapped_count, problems = records_to_csv_rows(records)
    out_path = FRONTIER_CSV_PATHS[model]

    with out_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FELDER_CSV)
        writer.writeheader()
        writer.writerows(csv_rows)

    all_frontier_dfs[model] = pd.DataFrame(csv_rows)
    print(f"{model}: CSV geschrieben → {out_path}")
    print(f"{model}: Schema-Verletzungs-Mappings angewendet: {mapped_count}")
    if problems:
        print(f"{model}: WARNUNGEN")
        for p in problems:
            print(f"  - {p}")

print("\nTerminal-Checks, falls validate.py vorhanden ist:")
for model, csv_path in FRONTIER_CSV_PATHS.items():
    print(f"  python annotation/validate.py {csv_path}")
if GOLD_PATH.exists():
    for model, csv_path in FRONTIER_CSV_PATHS.items():
        print(f"  python annotation/validate.py {GOLD_PATH} --kappa-against {csv_path}")



chatgpt: Records geladen: 12 (erwartet: 12)
chatgpt: CSV geschrieben → /home/jovyan/work/notebooks/Maschine learning/LLM-Workshop/annotation/frontier_gold_chatgpt.csv
chatgpt: Schema-Verletzungs-Mappings angewendet: 0

claude: Records geladen: 12 (erwartet: 12)
claude: CSV geschrieben → /home/jovyan/work/notebooks/Maschine learning/LLM-Workshop/annotation/frontier_gold_claude.csv
claude: Schema-Verletzungs-Mappings angewendet: 0

Terminal-Checks, falls validate.py vorhanden ist:
  python annotation/validate.py /home/jovyan/work/notebooks/Maschine learning/LLM-Workshop/annotation/frontier_gold_chatgpt.csv
  python annotation/validate.py /home/jovyan/work/notebooks/Maschine learning/LLM-Workshop/annotation/frontier_gold_claude.csv
  python annotation/validate.py /home/jovyan/work/notebooks/Maschine learning/LLM-Workshop/annotation/meine_gold.csv --kappa-against /home/jovyan/work/notebooks/Maschine learning/LLM-Workshop/annotation/frontier_gold_chatgpt.csv
  python annotation/validate.py

In [3]:
# κ-Compute: eigene Gold-Annotation ↔ ChatGPT / Claude
# Gleicher Grundalgorithmus wie validate.py, hier für beide Frontier-Dateien in einer Tabelle.

if gold_df is None:
    raise FileNotFoundError("Für den κ-Vergleich fehlt annotation/meine_gold.csv.")

def cohen_kappa(labels_a, labels_b):
    assert len(labels_a) == len(labels_b)
    n = len(labels_a)
    if n == 0:
        return float("nan")
    agreed = sum(1 for a, b in zip(labels_a, labels_b) if a == b)
    p_o = agreed / n
    cat_a, cat_b = Counter(labels_a), Counter(labels_b)
    p_e = sum((cat_a[c] / n) * (cat_b[c] / n) for c in set(cat_a) | set(cat_b))
    if p_e == 1.0:
        return float("nan")
    return (p_o - p_e) / (1 - p_e)

def norm_label(v):
    if pd.isna(v):
        return ""
    return str(v).strip()

mein_df = gold_df.rename(columns={"id": "refnr"})
mein_by_id = {str(r["refnr"]): dict(r) for _, r in mein_df.iterrows()}

CAT_FELDER = ["homeoffice", "vertragsart", "erfahrungslevel"]

kappa_rows = []
disagreements = []

for model, csv_path in FRONTIER_CSV_PATHS.items():
    frontier_df = pd.read_csv(csv_path).rename(columns={"id": "refnr"})
    frontier_by_id = {str(r["refnr"]): dict(r) for _, r in frontier_df.iterrows()}

    # Reihenfolge: Gold-Reihenfolge, nicht alphabetisch
    common_ids = [rid for rid in gold_ids if rid in frontier_by_id]
    if not common_ids:
        print(f"Warnung: Keine gemeinsamen IDs für {model}.")
        continue

    for feld in CAT_FELDER:
        a = [norm_label(mein_by_id[r].get(feld)) for r in common_ids]
        b = [norm_label(frontier_by_id[r].get(feld)) for r in common_ids]
        k = cohen_kappa(a, b)
        agree = sum(1 for x, y in zip(a, b) if x == y)
        kappa_rows.append({
            "modell": model,
            "feld": feld,
            "κ": round(k, 3),
            "Übereinst.": f"{agree/len(common_ids):.0%}",
            "agree": agree,
            "n": len(common_ids),
        })
        for rid, x, y in zip(common_ids, a, b):
            if x != y:
                disagreements.append({
                    "modell": model,
                    "refnr": rid,
                    "feld": feld,
                    "ich": x,
                    "frontier": y,
                })

kappa_df = pd.DataFrame(kappa_rows)
disagreement_df = pd.DataFrame(disagreements)

print("κ Mensch ↔ Frontier:\n")
display(kappa_df)

print("\nInterpretation (Landis & Koch): < 0.40 mäßig · 0.41–0.60 moderat · 0.61–0.80 substanziell · > 0.80 fast perfekt")
print(f"\nDisagreements gesamt über beide Modelle und kategoriale Felder: {len(disagreement_df)}")

if not disagreement_df.empty:
    display(disagreement_df.sort_values(["modell", "feld", "refnr"]).reset_index(drop=True))


κ Mensch ↔ Frontier:



,modell,feld,κ,Übereinst.,agree,n
0,chatgpt,homeoffice,0.507,75%,9,12
1,chatgpt,vertragsart,1.000,100%,12,12
2,chatgpt,erfahrungslevel,0.526,75%,9,12
3,claude,homeoffice,0.385,67%,8,12
4,claude,vertragsart,1.000,100%,12,12
5,claude,erfahrungslevel,0.363,50%,6,12



Interpretation (Landis & Koch): < 0.40 mäßig · 0.41–0.60 moderat · 0.61–0.80 substanziell · > 0.80 fast perfekt

Disagreements gesamt über beide Modelle und kategoriale Felder: 16


,modell,refnr,feld,ich,frontier
0,chatgpt,11949-17215590-S,erfahrungslevel,nicht_genannt,junior
1,chatgpt,13151-1570027-1-S,erfahrungslevel,mid,junior
2,chatgpt,14225-2aaa34ba5c393d2a-S,erfahrungslevel,nicht_genannt,mid
3,chatgpt,14225-2aaa34ba5c393d2a-S,homeoffice,teilweise,nein
4,chatgpt,16724-0062809539-S,homeoffice,ja,nicht_genannt
5,chatgpt,18896-8565435-S,homeoffice,teilweise,nicht_genannt
6,claude,11949-17196786-S,erfahrungslevel,mid,senior
7,claude,12265-489382_JB5131539-S,erfahrungslevel,mid,nicht_genannt
8,claude,13151-1568687-1-S,erfahrungslevel,mid,junior
9,claude,13151-1570027-1-S,erfahrungslevel,mid,junior


In [7]:
# Direkter Modellvergleich: ChatGPT ↔ Claude
# Hilft, die Anzeigen/Felder zu finden, bei denen Frontier-Modelle uneinig sind.

if not all(path.exists() for path in FRONTIER_CSV_PATHS.values()):
    raise FileNotFoundError("Bitte zuerst die JSON→CSV-Zelle ausführen.")

def norm_label_local(v):
    if pd.isna(v):
        return ""
    return str(v).strip()

chatgpt_df = pd.read_csv(FRONTIER_CSV_PATHS["chatgpt"]).rename(columns={"id": "refnr"})
claude_df = pd.read_csv(FRONTIER_CSV_PATHS["claude"]).rename(columns={"id": "refnr"})

chatgpt_by_id = {str(r["refnr"]): dict(r) for _, r in chatgpt_df.iterrows()}
claude_by_id = {str(r["refnr"]): dict(r) for _, r in claude_df.iterrows()}

compare_fields = ["homeoffice", "vertragsart", "erfahrungslevel", "gehalt_min_eur", "gehalt_zeitraum", "skills_top3"]
common_ids_models = [rid for rid in chatgpt_by_id if rid in claude_by_id]

model_diff_rows = []
for rid in common_ids_models:
    for feld in compare_fields:
        a = norm_label_local(chatgpt_by_id[rid].get(feld))
        b = norm_label_local(claude_by_id[rid].get(feld))
        if a != b:
            model_diff_rows.append({
                "refnr": rid,
                "feld": feld,
                "chatgpt": a,
                "claude": b,
            })

model_diff_df = pd.DataFrame(model_diff_rows)
print(f"Direkte ChatGPT-Claude-Differenzen: {len(model_diff_df)}")
if not model_diff_df.empty:
    display(model_diff_df.sort_values(["refnr", "feld"]).reset_index(drop=True))


Direkte ChatGPT-Claude-Differenzen: 14


,refnr,feld,chatgpt,claude
0,11949-17196786-S,erfahrungslevel,mid,senior
1,11949-17196786-S,skills_top3,Machine Learning|Python|statistische Modellierung,Python|Machine Learning
2,11949-17215590-S,erfahrungslevel,junior,nicht_genannt
3,12265-489382_JB5131539-S,erfahrungslevel,mid,nicht_genannt
4,12265-489382_JB5131539-S,skills_top3,SQL|ETL|Power BI,
5,13151-1568687-1-S,erfahrungslevel,mid,junior
6,13151-1570027-1-S,homeoffice,teilweise,nicht_genannt
7,13644-307612-S,erfahrungslevel,mid,nicht_genannt
8,14036-0005680f46a001-S,erfahrungslevel,mid,nicht_genannt
9,14225-2aaa34ba5c393d2a-S,erfahrungslevel,mid,nicht_genannt


### Drei konkrete Disagreement-Beispiele für das Memo

Nutze `disagreement_df` und `model_diff_df` aus den vorherigen Zellen.

Pro Beispiel: refnr, Feld, eigener Wert, Frontier-Wert, **wer hatte recht** mit Bezug auf Schema-Definition oder Anzeigen-Text.

---

**Disagreement 1 — refnr: _XXXXX_** (Feld: `_`)

- Ich: `_wert_`
- Frontier: `_wert_`
- **Wer hat recht?** _Bezug auf SCHEMA.md-Definition + konkrete Textstelle der Anzeige._

---

**Disagreement 2 — refnr: _XXXXX_** (Feld: `_`)

- Ich: `_`
- Frontier: `_`
- **Wer hat recht?** _

---

**Disagreement 3 — refnr: _XXXXX_** (Feld: `_`)

- Ich: `_`
- Frontier: `_`
- **Wer hat recht?** _

---

### Material fürs Memo (`memo_make_or_buy.md` im Repo-Root)

Aus der κ-Tabelle + den Disagreements:
- Auf welchen Feldern ist ChatGPT/Claude vertrauenswürdig (κ ≥ 0.6)?
- Auf welchen Feldern systematisch unsicher (κ < 0.4)?
- Wo sind sich ChatGPT und Claude uneinig?
- Wie viele der Disagreements waren „Frontier-Fehler" vs. „Schema-Lücke"?
- Würde ich die restlichen 78 Anzeigen vom Frontier annotieren lassen, selbst, oder hybrid?


In [5]:
# Optional: κ-Tabelle und Differenzen als CSV exportieren

EXPORT_DIR = ANNOTATION_DIR
if "kappa_df" in globals() and not kappa_df.empty:
    kappa_df.to_csv(EXPORT_DIR / "frontier_kappa_chatgpt_claude.csv", index=False, encoding="utf-8")
    print("geschrieben:", EXPORT_DIR / "frontier_kappa_chatgpt_claude.csv")

if "disagreement_df" in globals() and not disagreement_df.empty:
    disagreement_df.to_csv(EXPORT_DIR / "frontier_disagreements_vs_gold.csv", index=False, encoding="utf-8")
    print("geschrieben:", EXPORT_DIR / "frontier_disagreements_vs_gold.csv")

if "model_diff_df" in globals() and not model_diff_df.empty:
    model_diff_df.to_csv(EXPORT_DIR / "frontier_differences_chatgpt_vs_claude.csv", index=False, encoding="utf-8")
    print("geschrieben:", EXPORT_DIR / "frontier_differences_chatgpt_vs_claude.csv")


geschrieben: /home/jovyan/work/notebooks/Maschine learning/LLM-Workshop/annotation/frontier_kappa_chatgpt_claude.csv
geschrieben: /home/jovyan/work/notebooks/Maschine learning/LLM-Workshop/annotation/frontier_disagreements_vs_gold.csv
geschrieben: /home/jovyan/work/notebooks/Maschine learning/LLM-Workshop/annotation/frontier_differences_chatgpt_vs_claude.csv
